# FiTuna — NVIDIA/Linux verification notebook

One-click reproduction of FiTuna's measured search on a **real NVIDIA GPU
(Linux)** via Google Colab. Complements the macOS (Apple Silicon) results in
[`docs/RESULTS.md`](https://github.com/leeyunseokarchive/fituna/blob/main/docs/RESULTS.md).

**Before running:** Runtime → Change runtime type → **T4 GPU** (free tier works).

What this proves, on hardware the maintainers don't own:
1. `nvidia-smi`-based hardware auto-detection (`fituna detect-hw`)
2. The full measured search (quantize → perplexity gate → bench → minimal `-ngl`)
3. `--resume` cache reproducibility

Total runtime ≈ 20–30 min, most of it the llama.cpp CUDA build.

In [ ]:
# 1. Confirm the GPU runtime
!nvidia-smi

In [ ]:
%%bash
# 2. Build llama.cpp with CUDA (the slow cell, ~10-20 min)
set -e
if [ ! -x llama.cpp/build/bin/llama-bench ]; then
  git clone --depth 1 https://github.com/ggml-org/llama.cpp
  cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release > /dev/null
  cmake --build llama.cpp/build --target llama-quantize llama-bench llama-perplexity llama-cli -j2 2>&1 | tail -3
fi
llama.cpp/build/bin/llama-bench --help > /dev/null && echo "llama.cpp ready" 

In [ ]:
# 3. Install FiTuna from the public repository
!pip install -q git+https://github.com/leeyunseokarchive/fituna
!fituna --help | head -5

In [ ]:
# 4. Hardware auto-detection -- exercises the nvidia-smi parsing path
!fituna detect-hw

In [ ]:
# 5. Fetch the demo model (SmolLM2-135M F16, Apache 2.0) + quality corpus
!pip install -q huggingface_hub datasets
from huggingface_hub import hf_hub_download
hf_hub_download("bartowski/SmolLM2-135M-Instruct-GGUF",
                "SmolLM2-135M-Instruct-f16.gguf", local_dir=".")
from datasets import load_dataset
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
open("wikitext-2-raw-test.txt", "w").write("\n".join(ds["text"]))
print("model + corpus ready")

In [ ]:
# 6. The measured search (same spec as the macOS run in docs/RESULTS.md)
!time fituna run --model SmolLM2-135M-Instruct-f16.gguf \
  --target-tps 240 --max-quality-loss 5 --ctx 4096 \
  --quant Q8_0,Q6_K,Q5_K_M,Q4_K_M --ppl-chunks 32 \
  --quality-corpus wikitext-2-raw-test.txt \
  --llama-bin-dir llama.cpp/build/bin --out ./out --resume

In [ ]:
# 7. Reproducibility: identical re-run answers from cache in ~1s
!time fituna run --model SmolLM2-135M-Instruct-f16.gguf \
  --target-tps 240 --max-quality-loss 5 --ctx 4096 \
  --quant Q8_0,Q6_K,Q5_K_M,Q4_K_M --ppl-chunks 32 \
  --quality-corpus wikitext-2-raw-test.txt \
  --llama-bin-dir llama.cpp/build/bin --out ./out --resume

### Optional — 4B model (≈40 extra minutes)

Repeats the headline Qwen3-4B experiment on the T4. Uncomment to run.

In [ ]:
# from huggingface_hub import hf_hub_download
# hf_hub_download("unsloth/Qwen3-4B-Instruct-2507-GGUF",
#                 "Qwen3-4B-Instruct-2507-F16.gguf", local_dir=".")
# !time fituna run --model Qwen3-4B-Instruct-2507-F16.gguf \
#   --target-tps 30 --max-quality-loss 5 --ctx 4096 \
#   --quant Q8_0,Q6_K,Q5_K_M,Q4_K_M --ppl-chunks 32 \
#   --quality-corpus wikitext-2-raw-test.txt \
#   --llama-bin-dir llama.cpp/build/bin --out ./out4b --resume

### Recording results

Numbers differ from the macOS run — that is the point: FiTuna reports what
*this* hardware measures. To contribute your run to the project, open an
issue with the cell outputs of steps 4, 6 and 7.